The way to determine if the parameters for PSF resampler are correct is to check how the PSD before and after resampling compare.

This notebook explores the computation of spectra for both input and resampled.

# Definitions

In [ ]:
%matplotlib inline

In [ ]:
import datetime as dt
import logging
from collections.abc import Mapping
from dataclasses import asdict, dataclass

import healpix_geo
import healpix_geo.auto
import healpix_plot as hp
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import scipy.signal.windows
import tol_colors as tc
import xarray as xr
from healpix_analyse.fft_local import LocalFFT
from healpix_analyse.nufft import (
    get_bounds,
    normalize_coordinates,
    nufft,
    windowing_weights_polygon,
)
from healpix_geo.auto import Grid
from healpix_resample import PSFResampler
from healpix_resample.base import ResampleResults
from mpl_toolkits.axes_grid1 import make_axes_locatable
from numpy.typing import NDArray
from swot_healpix.analysis.spectra import FFTComputer, average_psd_azimuth
from swot_healpix.data import SWOT
from swot_healpix.utils import (
    add_look_arrows,
    format_latlon,
    get_cell_size,
    set_latlon_formatters,
)

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
level = 15
cell_size = get_cell_size(level)
healpix_grid = {"level": level, "indexing_scheme": "nested", "ellipsoid": "WGS84"}

## Load data

In [ ]:
# Load data
def load(cycle: int, track: int, lat: float, side: str = "left") -> xr.Dataset:
    """Load SWOT data."""
    swot = SWOT(cycle=cycle, track=track, version="PGC0")
    swot_file = swot.get_file()
    swath = xr.open_dataset(swot_file, group=side, chunks=-1)

    center_lat = swath.latitude.isel(num_pixels=swath.num_pixels.size // 2).values
    center_lat = np.sort(center_lat)  # just in case, just only modify slighty over land
    i_line = np.searchsorted(center_lat, lat)

    swath = swath.isel(num_lines=slice(i_line - 250, i_line + 250))

    if side == "left":
        swath = swath.isel(num_pixels=slice(None, None, -1))
    if swot.pass_type == "descending":
        swath = swath.isel(num_lines=slice(None, None, -1))

    swath.longitude.load()
    swath.latitude.load()
    swath.ssh_karin_2.load()

    swath = swath.dropna("num_lines", how="all", subset=["longitude"])
    swath = swath.dropna("num_pixels", how="all", subset=["longitude"])

    swath["ssh_karin_2"] -= swath.ssh_karin_2.mean()

    return swath

## Resample

In [ ]:
def resample(
    swath: xr.Dataset,
    cell_ids: NDArray,
    lam: float = 0.0,
    verbose: float = True,
    **kwargs,
) -> ResampleResults:
    """Resample SSH."""
    lon = swath.longitude.values.ravel()
    lat = swath.latitude.values.ravel()
    ssh = swath.ssh_karin_2.values.ravel()

    resampler = PSFResampler(
        lon_deg=lon, lat_deg=lat, level=level, out_cell_ids=cell_ids, **kwargs
    )
    resampled = resampler.resample(ssh, lam=lam)
    if verbose:
        print(
            f"{resampled.cg_niters} iterations, "
            f"final residual: {resampled.cg_residual_norms[-1]}"
        )
    return resampled

## Get pixel size

In [ ]:
def get_pixel_size(
    swath: xr.Dataset,
    cross_track_margin: int = 30,
    deg: bool = False,
    verbose: bool = True,
) -> tuple[float, float]:
    """Get average pixel size over swath. Avoid far end in cross-track."""
    lon = swath.longitude[:, :-cross_track_margin]
    lat = swath.latitude[:, :-cross_track_margin]

    # [(lon, lat), vertical/along-track, horizontal/cross-track]
    r = np.stack([lon, lat], axis=0)

    dr_x = np.diff(r, axis=2)
    dr_y = np.diff(r, axis=1)

    if not deg:
        factor = 6371e3 / 180 * np.pi
        dr_x *= factor
        dr_y *= factor

        # correct radius for longitude
        dr_x[0] *= np.cos(np.deg2rad(r[1, :, 1:]))
        dr_y[0] *= np.cos(np.deg2rad(r[1, 1:, :]))

    dx = np.mean(np.linalg.norm(dr_x, axis=0))
    dy = np.mean(np.linalg.norm(dr_y, axis=0))

    if verbose:
        unit = "°" if deg else "m"
        print(
            f"Resolution horizontal/cross-track: {dx:g}{unit}, "
            f"vertical/along-track: {dy:g}{unit}"
        )

    return dx, dy

## Get orientation

In [ ]:
def get_track_angle(swath: xr.Dataset) -> float:
    """Get track angle of SWOT swath."""
    center_line = swath.isel(num_pixels=swath.num_pixels.size // 2)
    dx = center_line.longitude.diff("num_lines")
    dy = center_line.latitude.diff("num_lines")
    return float(np.mean(np.atan2(dy, dx)))


def get_angles_north_aligned(track_angle: float, side: str) -> NDArray:
    """Return North, v_sat and look angles for a north aligned plot."""
    look = track_angle + np.pi / 2 if side == "left" else track_angle - np.pi / 2
    return np.asarray([np.pi / 2.0, track_angle, look])


def rotate_xy(angle_rad: float, x: NDArray, y: NDArray) -> tuple[NDArray, NDArray]:
    rot_mat = np.array(
        [
            [np.cos(angle_rad), -np.sin(angle_rad)],
            [np.sin(angle_rad), np.cos(angle_rad)],
        ]
    )

    xx, yy = np.meshgrid(x, y)
    xy = np.stack([xx, yy], axis=-1).reshape(-1, 2)
    rot_xy = (xy @ rot_mat).reshape([*xx.shape, 2])
    return rot_xy[:, :, 0], rot_xy[:, :, 1]

## Select span

In [ ]:
def select_span(swath: xr.Dataset, dx: float, dy: float) -> xr.Dataset:
    """Select a 40x40km window in the middle of the swath."""
    center_i_line = swath.num_lines.size // 2
    center_i_pixel = swath.num_pixels.size // 2
    nx = int(40e3 / dx)
    ny = int(40e3 / dy)
    iy0 = center_i_line - ny // 2
    iy1 = center_i_line + ny // 2
    ix0 = center_i_pixel - nx // 2
    ix1 = center_i_pixel + ny // 2

    if iy0 < 0 or iy1 > swath.num_lines.size or ix0 < 0 or ix1 > swath.num_pixels.size:
        raise IndexError

    return swath.isel(
        num_lines=slice(max(0, center_i_line - ny // 2), center_i_line + ny // 2),
        num_pixels=slice(max(0, center_i_pixel - nx // 2), center_i_pixel + nx // 2),
    )


def get_span_vertices(span: xr.Dataset) -> NDArray:
    """Return vertices of span bounding box."""
    # [(lon, lat), vertical/along-track, horizontal/cross-track]
    xy = np.stack([span.longitude, span.latitude], axis=0)
    return np.asarray(
        [
            xy[:, 0, 0],  # bottom left
            xy[:, 0, -1],  # bottom right
            xy[:, -1, -1],  # top right
            xy[:, -1, 0],  # top left
        ]
    )

## Compute spectra

In [ ]:
def compute_psd(
    ssh: NDArray, dx: float = 250, dy: float = 250
) -> tuple[NDArray, NDArray, NDArray]:
    """Compute 2D PSD."""
    ny, nx = ssh.shape
    fft_c = FFTComputer(
        box_size_x=nx, box_size_y=ny, dx=dx, dy=dy, apodizing_kind="tukey"
    )

    psd_2d = fft_c.psd(ssh)
    fx, fy = fft_c.get_frequencies()

    return fx, fy, psd_2d


def get_xy_meters(cell_ids: NDArray, grid: Mapping | Grid) -> tuple[NDArray, NDArray]:
    """Get cell coordinates in degrees or meters."""
    grid = Grid(**grid)
    lon, lat = healpix_geo.auto.healpix_to_lonlat(cell_ids, grid)
    x = (lon - lon[0]) * 6371e3 * np.pi / 180 * np.cos(np.deg2rad(lat))
    y = (lat - lat[0]) * 6371e3 * np.pi / 180
    return x, y

In [ ]:
def project_gnomonic(cell_ids: NDArray, data: NDArray) -> tuple[LocalFFT, NDArray]:
    transform = LocalFFT(cell_ids, level=level, ellipsoid="WGS84", device="cpu")
    grid = transform.project(data)
    mask = transform.coverage_mask.numpy()
    weighted = grid * mask

    return transform, weighted


def get_xy_gnomonic(transform: LocalFFT) -> tuple[NDArray, NDArray]:
    N = transform.grid_size
    dx = (np.arange(N) - N / 2) * transform.pixel_size_rad
    dx *= 180 / np.pi
    x = transform.centre_lon_deg + dx
    y = transform.centre_lat_deg + dx
    return x, y


def fft_gnomonic(
    transform: LocalFFT, data: NDArray
) -> tuple[NDArray, NDArray, NDArray]:
    fft = np.fft.fft2(data, norm="ortho")
    f = np.fft.fftfreq(transform.grid_size, d=transform.pixel_size_rad * 2) / 6371
    f = np.fft.fftshift(f)

    return f, f, fft


def average_psd_azimuth_gnomonic(
    transform: LocalFFT, fft: NDArray
) -> tuple[NDArray, NDArray]:
    f, psd_1d = transform.ps(fft)
    return f / 6371, psd_1d

## Study cases

In [ ]:
@dataclass
class StudyCase:
    name: str
    cycle: int
    track: int
    lat: float
    date: dt.date
    side: str = "left"

    def load(self) -> xr.Dataset:
        """Load part of the SWOT swath for this case."""
        return load(cycle=self.cycle, track=self.track, lat=self.lat, side=self.side)

cases_def = [
    StudyCase("long_swells", 546, 19, lat=-46.96, date=dt.date(2023, 6, 9)),
    StudyCase("remote_reflections", 547, 21, lat=-45.4, date=dt.date(2023, 6, 10)),
    StudyCase("crossing_swells", 548, 17, lat=-29.3, date=dt.date(2023, 6, 11)),
]

for case in cases_def:
    case.swath = case.load()
    case.track_angle = get_track_angle(case.swath)


def get_cases() -> list[StudyCase]:
    cases = [StudyCase(**asdict(case)) for case in cases_def]
    for case, case_def in zip(cases, cases_def, strict=True):
        case.swath = case_def.swath
        case.track_angle = case_def.track_angle
    return cases

cases = get_cases()

## All in one

In [ ]:
def all_in_one(
    swath: xr.Dataset,
    lam: float = 0.0,
    sigma_m: float | None = None,
    threshold: float = 0.1,
    verbose: bool = True,
) -> xr.Dataset:
    """Do all processing at once. Return 1D PSD."""
    dx, dy = get_pixel_size(swath, verbose=False)
    span = select_span(swath, dx, dy)
    span_vertices = get_span_vertices(span)

    cell_ids, _, _ = healpix_geo.nested.polygon_coverage(
        span_vertices, level, ellipsoid="WGS84"
    )
    resampled = resample(
        span, cell_ids, lam=lam, threshold=threshold, sigma_m=sigma_m, verbose=verbose
    )
    cell_ids = resampled.cell_ids
    ssh_hp = resampled.cell_data

    cell_lon, cell_lat = healpix_geo.nested.healpix_to_lonlat(
        cell_ids, depth=level, ellipsoid="WGS84"
    )
    weights = apodize_vertices(cell_lon, cell_lat, span_vertices, length=0.07)

    fx, fy, psd_2d = compute_psd(span.ssh_karin_2.values, dx, dy)
    fx *= 1000
    fy *= 1000
    fr, psd_1d = average_psd_azimuth(psd_2d, fx, fy)

    fx_hp, fy_hp, fhat = nfft(
        cell_ids,
        (ssh_hp - np.mean(ssh_hp)) * weights,
        grid=healpix_grid,
        n_frequencies=[200, 200],
    )
    fx_hp *= 1000
    fy_hp *= 1000
    psd_2d_hp = np.abs(fhat) ** 2
    fr_hp, psd_1d_hp = average_psd_azimuth(psd_2d_hp, fx_hp, fy_hp)

    ds = xr.Dataset(
        {
            "psd": (("fr"), psd_1d),
            "psd_hp": (("fr_hp"), psd_1d_hp),
        },
        coords={
            "fr": fr,
            "fr_hp": fr_hp,
        },
    )
    return ds

# Single image

First case study (long-swells).

## Load data

In [ ]:
case = cases_def[0]
swath = case.swath

fig, ax = plt.subplots(figsize=(4, 3), layout="constrained", dpi=200)
swath.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    center=False,
    cmap="Blues_r",
    add_labels=False,
    cbar_kwargs={"pad": 0.02, "label": "[m]"},
)

add_look_arrows(
    ax,
    *get_angles_north_aligned(case.track_angle, case.side),
    label_offsets={"look": (8, 4)},
    box_kwargs={"pad": 0.8},
)

set_latlon_formatters(ax)
ax.set_aspect("equal")

## Find SWOT pixel size

We need it to select a region of the same size as our healpix diamond.
We also need it to compute frequencies accurately.


In [ ]:
cross_track_margin = 30
# Avoid that many pixels on the close edge cross-track when averaging pixel size

# [(lon, lat), along-track, cross-track]
r = np.stack([swath.longitude, swath.latitude], axis=0)

factor = 6371e3 / 180 * np.pi
dr_x = np.diff(r, axis=2) * factor
dr_y = np.diff(r, axis=1) * factor

# correct radius for longitude
dr_x[0] *= np.cos(np.deg2rad(r[1, :, 1:]))
dr_y[0] *= np.cos(np.deg2rad(r[1, 1:, :]))

dr_x = np.linalg.norm(dr_x, axis=0)
dr_y = np.linalg.norm(dr_y, axis=0)

In [ ]:
fig, axes = plt.subplots(
    1, 2, sharex=True, sharey=True, layout="constrained", figsize=(3.7, 3.7), dpi=200
)
labels = (["Δ(Cross-track)", "width"], ["Δ(Along-track)", "height"])

for ax, dr, (title, unit) in zip(axes, [dr_x, dr_y], labels):
    vmin = np.quantile(dr, 0.01)
    vmax = np.quantile(dr, 0.99)
    im = ax.imshow(dr, origin="lower", vmin=vmin, vmax=vmax, cmap=tc.rainbow_PuRd)

    divider = make_axes_locatable(ax)
    ax_avg_x = divider.append_axes("top", 0.42, pad=0.05, sharex=ax)
    ax_avg_y = divider.append_axes("right", 0.35, pad=0.05, sharey=ax)
    ax_avg_x.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)
    ax_avg_y.tick_params(labelbottom=False, labelleft=False, left=False, bottom=False)

    ax_avg_x.plot(np.mean(dr, axis=0), lw=0.7)
    ax_avg_y.plot(np.mean(dr, axis=1), np.arange(dr.shape[0]), lw=0.7)

    ax_avg_x.set_title(title)

    for ax_ in (ax, ax_avg_x):
        ax_.axvline(
            dr.shape[1] - cross_track_margin, lw=0.7, alpha=0.7, color=tc.bright.red
        )

    fig.colorbar(im, ax=ax, location="bottom", label=f"Pixel {unit}")
axes[0].set_ylabel("Along-track")

In [ ]:
fig, axes = plt.subplots(
    2, 1, sharex=True, figsize=(4, 2.3), dpi=200, layout="constrained"
)
fig.suptitle("Along-track average")

for ax, dr, label in zip(
    axes, [dr_x, dr_y], ("width\n(cross-track)", "height\n(along-track)")
):
    ax.set_ylabel("pixel " + label)
    ax.plot(np.mean(dr, axis=0))
    ax.axvline(
        dr.shape[1] - cross_track_margin, color=tc.bright.red, alpha=0.7
    )
    ax.axhline(np.mean(dr[:, :-cross_track_margin]), color="k")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f"{x:.1f}"))

axes[1].set_xlabel("Cross-track pixel index")

In [ ]:
dx, dy = get_pixel_size(swath, cross_track_margin=cross_track_margin)
dx_deg, dy_deg = get_pixel_size(swath, cross_track_margin=cross_track_margin, deg=True)

## Select span

with the same size (in meters) as the healpix diamond.

In [ ]:
span = select_span(swath, dx, dy)
span_vertices = get_span_vertices(span)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3), layout="constrained", dpi=200)

im = swath.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    center=False,
    cmap="Blues_r",
    add_labels=False,
    cbar_kwargs={"pad": 0.02, "label": "[m]"},
)

ax.add_artist(plt.Polygon(span_vertices, ec=tc.bright.red, fc="none"))

add_look_arrows(
    ax,
    *get_angles_north_aligned(case.track_angle, case.side),
    label_offsets={"look": (8, 4)},
    box_kwargs={"pad": 0.8},
)

set_latlon_formatters(ax)

ax.set_aspect("equal")

## Resample

and reshape to 2D

In [ ]:
cell_ids, _, _ = healpix_geo.nested.polygon_coverage(
    span_vertices, depth=level, ellipsoid="WGS84"
)

resampled = resample(swath, cell_ids, sigma_m=275, lam=0.2)
cell_ids = resampled.cell_ids
ssh_hp = resampled.cell_data

cell_lon, cell_lat = healpix_geo.nested.healpix_to_lonlat(
    cell_ids, depth=level, ellipsoid="WGS84"
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(5, 2),
    layout="constrained",
    sharex=True,
    sharey=True,
    dpi=150,
)
ax1, ax2 = axes

vmin = float(span.ssh_karin_2.min())
vmax = float(span.ssh_karin_2.max())

im = span.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    vmin=vmin,
    vmax=vmax,
    ax=ax1,
    add_labels=False,
    add_colorbar=False,
    cmap="Blues_r",
)
ax1.set_title("Input span")

ims = hp.pcolormesh(
    cell_ids,
    ssh_hp,
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=vmin,
    vmax=vmax,
    cmap="Blues_r",
    ax=ax2,
)
ax2.set_title("Healpix")
fig.colorbar(im, ax=axes, label="[m]")

for ax in axes:
    ax.set_aspect("equal")
    ax.grid(True)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))
    set_latlon_formatters(ax)

## Spectra

### SWOT-aligned

In [ ]:
# SWOT
fx, fy, psd_2d = compute_psd(span.ssh_karin_2.values, dx, dy)
fx *= 1000
fy *= 1000
fr, psd_1d = average_psd_azimuth(psd_2d, fx, fy)

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", figsize=(3.8, 3), dpi=200)
ax1, ax2 = axes

x = np.arange(span.num_pixels.size) * dx / 1e3
y = np.arange(span.num_lines.size) * dy / 1e3
win_x = scipy.signal.windows.tukey(x.size, sym=True)
win_y = scipy.signal.windows.tukey(y.size, sym=True)
win = win_y[:, None] * win_x

im1 = ax1.pcolormesh(x, y, span.ssh_karin_2.values * win, cmap="Blues_r")
im2 = ax2.pcolormesh(fx, fy, psd_2d, norm="log")
fig.colorbar(im1, ax=ax1, location="top", pad=0.02, label="[m]")
fig.colorbar(im2, ax=ax2, location="top", pad=0.02)

ax1.set_xlabel("[km]")
ax2.set_xlabel("[cycle/km]")

for ax in axes:
    ax.set_aspect("equal")

### NFFT

In [ ]:
# Healpix weights for windowing

weights = windowing_weights_polygon(cell_lon, cell_lat, span_vertices, length=0.07)

fig, axes = plt.subplots(
    2, 1, layout="constrained", figsize=(3, 3), dpi=200, sharex=True, sharey=True
)
ax1, ax2 = axes

ims = hp.pcolormesh(
    cell_ids,
    weights,
    healpix_grid=healpix_grid,
    level_tile=7,
    ax=ax1,
)
ax1.add_artist(plt.Polygon(span_vertices, ec=tc.bright.red, fc="none"))
fig.colorbar(ims[0], ax=ax1, pad=0.02, label="windowing weights")

ims = hp.pcolormesh(
    cell_ids,
    ssh_hp * weights,
    healpix_grid=healpix_grid,
    level_tile=7,
    vmin=-0.6,
    vmax=0.6,
    cmap="Blues_r",
    ax=ax2,
)
fig.colorbar(ims[0], ax=ax2, pad=0.02, label="weighted SSH [m]")

for ax in axes:
    ax.set_aspect("equal")
    set_latlon_formatters(ax)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

In [ ]:
xy = get_xy_meters(cell_ids, grid=healpix_grid)
fx_nfft, fy_nfft, fhat = nufft(
    *xy,
    (ssh_hp - np.mean(ssh_hp)) * weights,
    n_frequencies=[200, 200],
)

psd_2d_nfft = np.abs(fhat)**2
fx_nfft *= 1000
fy_nfft *= 1000

fr_nfft, psd_1d_nfft = average_psd_azimuth(psd_2d_nfft, fx_nfft, fy_nfft)

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", figsize=(4, 2.3), dpi=200)
ax1, ax2 = axes

ims = hp.pcolormesh(
    cell_ids, ssh_hp, healpix_grid=healpix_grid, level_tile=7, cmap="Blues_r", ax=ax1
)
fig.colorbar(ims[0], ax=ax1, location="top", pad=0.02, label="[m]")

im = ax2.pcolormesh(fx_nfft, fy_nfft, psd_2d_nfft, norm="log")
fig.colorbar(im, ax=ax2, location="top", pad=0.02)

ax2.set_xlabel("[cycle/km]")
set_latlon_formatters(ax1)
for ax in axes:
    ax.set_aspect("equal")

In [ ]:
fig, ax = plt.subplots(layout="constrained", figsize=(2.5, 1.2), dpi=200)

ims = hp.pcolormesh(
    cell_ids, ssh_hp, healpix_grid=healpix_grid, level_tile=7, cmap="Blues_r", ax=ax
)
fig.colorbar(ims[0], ax=ax, location="right", pad=0.02, label="[m]")
ax.set_title("SWOT SSH", fontsize=7)
set_latlon_formatters(ax)
ax.xaxis.set_major_locator(mtick.MaxNLocator(4))
ax.set_aspect("equal")

fig, axes = plt.subplots(
    1, 2, layout="constrained", figsize=(4, 2.8), dpi=200, sharex=True, sharey=True
)
ax1, ax2 = axes

im = ax1.pcolormesh(
    *rotate_xy(np.pi / 2 - case.track_angle, fx, fy), psd_2d, norm="log"
)
fig.colorbar(im, ax=ax1, location="top", pad=0.02, label="SWOT aligned FFT")
ax1.set_xlim(*fx_nfft[[0, -1]])
ax1.set_ylim(*fy_nfft[[0, -1]])

im = ax2.pcolormesh(fx_nfft, fy_nfft, psd_2d_nfft, norm="log")
fig.colorbar(im, ax=ax2, location="top", pad=0.02, label="HEALPix NFFT")

add_look_arrows(
    ax1,
    *get_angles_north_aligned(case.track_angle, case.side),
    label_offsets={"look": (8, 4)},
    box_kwargs={"pad": 0.8},
    box_size=0.15,
    arrows_kwargs={"width": 1.7},
    label_kwargs={"north": {"ha": "center"}},
)

ax1.set_ylabel("[$km^{-1}$]")
for ax in axes:
    ax.set_xlabel("[$km^{-1}$]")
    ax.set_aspect("equal")

### Gnomonic

In [ ]:
data = ssh_hp
data = data - np.mean(data)
data *= weights

transform, weighted = project_gnomonic(cell_ids, data)
x_gnom, y_gnom = get_xy_gnomonic(transform)

fx_gnom, fy_gnom, fft = fft_gnomonic(transform, weighted)
psd_2d_gnom = np.abs(fft)**2
fr_gnom, psd_1d_gnom = average_psd_azimuth_gnomonic(transform, fft)

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", figsize=(4, 2.3), dpi=200)
ax1, ax2 = axes

im1 = ax1.pcolormesh(
    x_gnom, y_gnom, weighted, cmap="Blues_r", vmin=-0.6, vmax=0.6
)
fig.colorbar(im1, ax=ax1, location="top", pad=0.02, label="[m]")

im2 = ax2.pcolormesh(fx_gnom, fy_gnom, psd_2d_gnom, norm="log")
fig.colorbar(im2, ax=ax2, location="top", pad=0.02)

ax2.add_artist(
    plt.Circle(
        (0, 0),
        radius=1.0,
        lw=0.7,
        ec="w",
        alpha=0.4,
        fc="none",
    )
)

for ax in axes:
    ax.set_aspect("equal")

### Method comparison

In [ ]:
fig, axes = plt.subplots(
    1, 3, layout="constrained", figsize=(6, 3.5), dpi=200, sharex=True, sharey=True
)
ax1, ax2, ax3 = axes

ax1.pcolormesh(*rotate_xy(np.pi/2-case.track_angle, fx, fy), psd_2d, norm="log")
ax1.set_title("SWOT aligned")

ax2.pcolormesh(fx_nfft, fy_nfft, psd_2d_nfft, norm="log")
ax2.set_title("NFFT")

ax3.pcolormesh(fx_gnom, fy_gnom, psd_2d_gnom, norm="log")
ax3.set_title("Gnomonic")

ax1.set_xlim(*fx_nfft[[0, -1]])
ax1.set_ylim(*fy_nfft[[0, -1]])

for ax in axes:
    ax.set_aspect("equal")
    ax.grid(False)

    for dist in [1000]:
        ax.add_artist(
            plt.Circle(
                (0, 0),
                radius=1 / (dist / 1000),
                lw=0.7,
                ec="w",
                alpha=0.4,
                fc="none",
            )
        )

ax1.set_ylabel("[cycle/km]")

add_look_arrows(
    ax1,
    *get_angles_north_aligned(case.track_angle, case.side),
    box_size=0.15,
    box_kwargs={"pad": 2},
    label_offsets={"look": (2, 2)},
    label_kwargs={"vsat": {"ha": "left"}},
)

In [ ]:
def plot(ax, fr, psd, factor=1, **kwargs) -> None:
    norm = np.sum(fr * psd)
    ax.plot(fr, psd / norm * factor, **kwargs)


fig, ax = plt.subplots(layout="constrained", figsize=(3, 1.8), dpi=200)
plot(ax, fr, psd_1d, label="SWOT")
plot(ax, fr_nfft, psd_1d_nfft, label="NFFT")
plot(ax, fr_gnom, psd_1d_gnom, factor=1.5, label="Gnomonic")
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("[cycle/km]")

### Test SWOT aligned NFFT

The SWOT aligned spectrum has lower power in low frequencies (around 10km period) compared to Healpix. Hypothesis: SWOT pixels are not space regularly (the exact position of pixels is correlated to the signal), so a basic FFT (as I used) may underestimate some frequencies.

Since NFFT seems to work well I test it directly on SWOT data.

In [ ]:
import finufft
from scipy.signal.windows import tukey
from swot_healpix.analysis.spectra.nufft import (
    get_bounds,
    get_frequencies,
    normalize_coordinates,
)

lon = span.longitude.values.ravel()
lat = span.latitude.values.ravel()
ssh = span.ssh_karin_2.values
n_frequencies = ssh.shape
My, Mx = ssh.shape
M = Mx * My

w = tukey(My)[:, None] * tukey(Mx)[None, :]

data = (ssh - np.mean(ssh)) * w

x = (lon - lon[0]) * 6371e3 * np.pi / 180 * np.cos(np.deg2rad(lat))
y = (lat - lat[0]) * 6371e3 * np.pi / 180
bounds = get_bounds(x, y)
xy_norm = normalize_coordinates((x, y), bounds)
fx_nfft_swot, fy_nfft_swot = get_frequencies(n_frequencies, bounds)
fx_nfft_swot *= 1000
fy_nfft_swot *= 1000

x_norm, y_norm = xy_norm
x_norm = x_norm.reshape((My, Mx))
y_norm = y_norm.reshape((My, Mx))

fhat = finufft.nufft2d1(
    x_norm.ravel(), y_norm.ravel(), data.ravel().astype("complex128"), n_frequencies
).T
psd_2d_nfft_swot = np.abs(fhat) ** 2

fr_nfft_swot, psd_1d_nfft_swot = average_psd_azimuth(
    psd_2d_nfft_swot, fx_nfft_swot, fy_nfft_swot
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(5.5, 3), layout="constrained", dpi=200)
ax1, ax2, ax3 = axes


im = ax1.pcolormesh(x_norm, y_norm, data)
fig.colorbar(im, ax=ax1, location="top", pad=0.02)
im = ax2.pcolormesh(fx_nfft_swot, fy_nfft_swot, psd_2d_nfft_swot, norm="log")
fig.colorbar(im, ax=ax2, location="top", pad=0.02, label="NFFT")
im = ax3.pcolormesh(*rotate_xy(np.pi/2-case.track_angle, fx, fy), psd_2d, norm="log")
fig.colorbar(im, ax=ax3, location="top", pad=0.02, label="FFT")

ax3.set_xlim(*ax2.get_xlim())
ax3.set_ylim(*ax2.get_ylim())
for ax in axes:
    ax.set_aspect("equal")

In [ ]:
def plot(ax, fr, psd, factor=1, **kwargs) -> None:
    norm = np.sum(fr * psd)
    ax.plot(fr, psd / norm * factor, **kwargs)


fig, ax = plt.subplots(layout="constrained", figsize=(3, 1.8), dpi=200)
plot(ax, fr, psd_1d, label="SWOT")
plot(ax, fr_nfft, psd_1d_nfft, label="NFFT")
plot(ax, fr_gnom, psd_1d_gnom, factor=1.5, label="Gnomonic")
plot(
    ax, fr_nfft_swot, psd_1d_nfft_swot, color="k", ls="--", label="SWOT NFFT"
)
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("[cycle/km]")

# Study cases

Use 3 different study cases (selected from Ardhuin 2024).

In [ ]:
cases = get_cases()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8), dpi=250, layout="constrained")
fig.suptitle("Input data")

for ax, c in zip(axes.flat, cases):
    im = c.swath.ssh_karin_2.plot.pcolormesh(
        x="longitude",
        y="latitude",
        add_labels=False,
        center=False,
        cbar_kwargs={"pad": 0.02},
        cmap="Blues_r",
        ax=ax,
    )
    ax.set_title(f"{c.name}\n{c.cycle}:{c.track}, {format_latlon(c.lat, 'lat')}")
    ax.set_aspect("equal")
    set_latlon_formatters(ax)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

    add_look_arrows(
        ax,
        *get_angles_north_aligned(c.track_angle, c.side),
        arrows_kwargs={"width": 1.5},
        label_kwargs={"north": {"ha": "center"}, "vsat": {"va": "center"}},
        label_offsets={"look": (8, 4)},
        box_kwargs={"pad": 0.8},
    )

im.colorbar.set_label("[m]")

In [ ]:
# Select span, resample
for case in cases:
    case.dx, case.dy = get_pixel_size(case.swath)
    case.span = select_span(case.swath, case.dx, case.dy)
    case.span_vertices = get_span_vertices(case.span)

    case.cell_ids, _, _ = healpix_geo.nested.polygon_coverage(
        case.span_vertices, level, ellipsoid="WGS84"
    )
    case.resampled = resample(
        case.span, case.cell_ids, lam=0.2, threshold=0.1, sigma_m=275
    )
    case.ssh_hp = case.resampled.cell_data

    case.cell_lon, case.cell_lat = healpix_geo.nested.healpix_to_lonlat(
        case.cell_ids, depth=level, ellipsoid="WGS84"
    )

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(6, 4), dpi=250, layout="constrained", sharex="col", sharey="col"
)
fig.suptitle("Selection for spectra")

for axes_col, c in zip(axes.T, cases):
    ax1, ax2 = axes_col
    vmin = float(c.span.ssh_karin_2.min())
    vmax = float(c.span.ssh_karin_2.max())
    kwargs = {"cmap": "Blues_r", "vmin": vmin, "vmax": vmax}

    im = ax1.pcolormesh(c.span.longitude, c.span.latitude, c.span.ssh_karin_2, **kwargs)
    ims = hp.pcolormesh(
        c.cell_ids, c.ssh_hp, healpix_grid=healpix_grid, level_tile=7, **kwargs, ax=ax2
    )
    fig.colorbar(im, ax=axes_col, location="bottom")
    ax1.set_title(c.name)

    for ax in axes_col:
        ax.set_aspect("equal")
        set_latlon_formatters(ax)
        ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

for ax, label in zip(axes[:, -1], ["Input", "Resampled"]):
    ax.annotate(
        label,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=90,
        weight="bold",
    )

In [ ]:
# Compute PSD
for case in cases:
    print(case)

    # SWOT Aligned
    case.fx, case.fy, case.psd_2d = compute_psd(
        case.span.ssh_karin_2.values, case.dx, case.dy
    )
    case.fx *= 1000
    case.fy *= 1000
    case.fr, case.psd_1d = average_psd_azimuth(case.psd_2d, case.fx, case.fy)

    # NFFT
    case.weights = windowing_weights_polygon(
        case.cell_lon, case.cell_lat, case.span_vertices, length=0.07
    )
    case.weighted = (case.ssh_hp - np.mean(case.ssh_hp)) * case.weights

    xy = get_xy_meters(case.cell_ids, healpix_grid)
    
    case.fx_nfft, case.fy_nfft, fhat = nufft(
        *xy,
        case.weighted,
        n_frequencies=[200, 200],
    )
    case.fx_nfft *= 1000
    case.fy_nfft *= 1000
    case.psd_2d_nfft = np.abs(fhat) ** 2
    case.fr_nfft, case.psd_1d_nfft = average_psd_azimuth(
        case.psd_2d_nfft, case.fx_nfft, case.fy_nfft
    )

    # Gnomonic
    case.transform, case.weighted_gnom = project_gnomonic(case.cell_ids, case.weighted)
    case.x_gnom, case.y_gnom = get_xy_gnomonic(case.transform)
    case.fx_gnom, case.fy_gnom, fft = fft_gnomonic(case.transform, case.weighted_gnom)
    case.psd_2d_gnom = np.abs(fft) ** 2
    case.fr_gnom, case.psd_1d_gnom = average_psd_azimuth_gnomonic(case.transform, fft)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(6, 4), dpi=250, layout="constrained")

for axes_col, case in zip(axes.T, cases, strict=False):
    ax1, ax2 = axes_col

    x = np.arange(case.span.num_pixels.size) * case.dx / 1e3
    y = np.arange(case.span.num_lines.size) * case.dy / 1e3

    win_x = scipy.signal.windows.tukey(case.span.num_pixels.size, sym=True)
    win_y = scipy.signal.windows.tukey(case.span.num_lines.size, sym=True)
    win = win_y[:, None] * win_x

    im1 = ax1.pcolormesh(x, y, case.span.ssh_karin_2.values * win, cmap="Blues_r")
    im2 = ax2.pcolormesh(case.fx, case.fy, case.psd_2d, norm="log")

    ax1.set_title(case.name)

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(case.track_angle, case.side),
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets={"look": (8, 4)},
        label_kwargs={"vsat": {"ha": "left"}, "north": {"ha": "right"}},
    )

    ax2.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

    for ax in axes_col:
        ax.set_aspect("equal")

axes[0, 0].set_ylabel("[km]")
axes[1, 0].set_ylabel("[cycle/km]")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(6, 3.5), dpi=250, layout="constrained")

for axes_col, case in zip(axes.T, cases, strict=False):
    ax1, ax2 = axes_col

    ims = hp.pcolormesh(
        case.cell_ids,
        case.weighted,
        healpix_grid=healpix_grid,
        level_tile=7,
        cmap="Blues_r",
        ax=ax1)
    im2 = ax2.pcolormesh(case.fx_nfft, case.fy_nfft, case.psd_2d_nfft, norm="log")

    ax1.set_title(case.name)

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(case.track_angle, case.side),
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets={"look": (8, 4)},
        label_kwargs={"vsat": {"ha": "left"}, "north": {"ha": "right"}},
    )

    ax2.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

    set_latlon_formatters(ax1)
    ax1.xaxis.set_major_locator(mtick.MaxNLocator(3))
    for ax in axes_col:
        ax.set_aspect("equal")

axes[1, 0].set_ylabel("[cycle/km]")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(6, 3.7), dpi=250, layout="constrained")

for axes_col, case in zip(axes.T, cases, strict=False):
    ax1, ax2 = axes_col

    ax1.pcolormesh(case.x_gnom, case.y_gnom, case.weighted_gnom, cmap="Blues_r")
    im2 = ax2.pcolormesh(case.fx_gnom, case.fy_gnom, case.psd_2d_gnom, norm="log")

    ax1.set_title(case.name)

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(case.track_angle, case.side),
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets={"look": (8, 4)},
        label_kwargs={"vsat": {"ha": "left"}, "north": {"ha": "right"}},
    )

    ax2.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

    set_latlon_formatters(ax1)
    ax1.xaxis.set_major_locator(mtick.MaxNLocator(3))
    for ax in axes_col:
        ax.set_aspect("equal")

axes[1, 0].set_ylabel("[cycle/km]")

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(6, 4), dpi=250, layout="constrained", sharex="col", sharey="col"
)

for axes_col, c in zip(axes.T, cases, strict=False):
    ax1, ax2 = axes_col

    im = ax1.pcolormesh(
        *rotate_xy(np.pi / 2 - c.track_angle, c.fx, c.fy), c.psd_2d, norm="log"
    )
    ax2.pcolormesh(c.fx_nfft, c.fy_nfft, c.psd_2d_nfft, norm="log")

    ax1.set_title(c.name)
    ax2.set_xlabel("[cycle/km]")

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(c.track_angle, c.side),
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets={"look": (8, 4)},
        label_kwargs={"vsat": {"ha": "left"}, "north": {"ha": "right"}},
    )

    ax2.set_xlim(*c.fx_nfft[[0, -1]])
    ax2.set_ylim(*c.fy_nfft[[0, -1]])

    for ax in axes_col:
        ax.set_aspect("equal")

for ax, label in zip(axes[:, -1], ["SWOT aligned", "NFFT"], strict=False):
    ax.annotate(
        label,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=90,
        weight="bold",
    )
    ax1.set_title(c.name)
    ax2.set_xlabel("[cycle/km]")

for ax in axes.flat:
    ax.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

In [ ]:
fig, axes = plt.subplots(
    3, 3, figsize=(6, 6), dpi=250, layout="constrained", sharex="col", sharey="col"
)

for axes_col, c in zip(axes.T, cases, strict=False):
    ax1, ax2, ax3 = axes_col

    im = ax1.pcolormesh(
        *rotate_xy(np.pi / 2 - c.track_angle, c.fx, c.fy), c.psd_2d, norm="log"
    )
    ax2.pcolormesh(c.fx_nfft, c.fy_nfft, c.psd_2d_nfft, norm="log")
    ax3.pcolormesh(c.fx_gnom, c.fy_gnom, c.psd_2d_gnom, norm="log")

    ax1.set_title(c.name)
    ax3.set_xlabel("[cycle/km]")

    add_look_arrows(
        ax1,
        *get_angles_north_aligned(c.track_angle, c.side),
        box_size=0.12,
        box_kwargs={"pad": 1},
        label_offsets={"look": (8, 4)},
        label_kwargs={"vsat": {"ha": "left"}, "north": {"ha": "right"}},
    )

    ax2.set_xlim(*c.fx_nfft[[0, -1]])
    ax2.set_ylim(*c.fy_nfft[[0, -1]])

    for ax in axes_col:
        ax.set_aspect("equal")

for ax, label in zip(axes[:, -1], ["SWOT aligned", "NFFT", "Gnomonic"], strict=False):
    ax.annotate(
        label,
        xy=(1, 0.5),
        xycoords="axes fraction",
        xytext=(3, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        rotation=90,
        weight="bold",
    )
    ax1.set_title(c.name)
    ax2.set_xlabel("[cycle/km]")

for ax in axes.flat:
    ax.add_artist(
        plt.Circle(
            (0, 0),
            radius=1.0,
            lw=0.7,
            ec="w",
            alpha=0.4,
            fc="none",
        )
    )

In [ ]:
fig, axes = plt.subplots(
    3, 1, figsize=(4, 4), dpi=300, layout="constrained", sharex=True
)

cset = tc.bright

for ax, c in zip(axes, cases):
    ax.yaxis.set_label_position("right")
    ax.set_ylabel(c.name)
    ax.plot(c.fr, c.psd_1d / 3e4, color=cset.blue, label="SWOT aligned")
    ax.plot(c.fr_nfft, c.psd_1d_nfft, color=cset.red, label="NFFT")
    ax.plot(c.fr_gnom, c.psd_1d_gnom, color=cset.green, label="Gnomonic")
    ax.set_xscale("log")
    ax.set_yscale("log")
axes[0].legend()
axes[-1].set_xlabel("[$km^{-1}$]")

# Varying parameters

## Sigma

In [ ]:
swath = cases[0].load()

lam = 0.5
psd_list = []
for sigma in [130, 175, 198, 250, 300, 400]:
    psd = all_in_one(swath, lam=lam, sigma_m=sigma)
    psd["psd_hp"] = psd.psd_hp.expand_dims(sigma=[sigma])
    psd_list.append(psd)
psd = xr.concat(psd_list, "sigma", data_vars=None)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5), dpi=200)
fig.suptitle(rf"Varying σ ($\lambda = {lam})$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(psd.fr, psd.psd/5e4, color="k", label="Input")
for sigma in psd.sigma.values:
    ax.plot(
        psd.fr_hp, psd.psd_hp.sel(sigma=sigma), lw=0.9, label=rf"$\sigma = {sigma}$"
    )
ax.legend()

## Lambda

In [ ]:
swath = cases[0].load()
sigma_m = 300
psd_list = []
for lam in [0, 0.01, 0.1, 0.5, 1, 10]:
    psd = all_in_one(swath, lam=lam, sigma_m=sigma_m)
    psd["psd_hp"] = psd.psd_hp.expand_dims(lam=[lam])
    psd_list.append(psd)
psd = xr.concat(psd_list, "lam", data_vars=None)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.5), dpi=200)
fig.suptitle(rf"Varying λ ($\sigma = {sigma_m})$")

ax.set_xscale("log")
ax.set_yscale("log")
ax.plot(psd.fr, psd.psd/5e4, color="k", label="Input")
for lam in psd.lam.values:
    ax.plot(psd.fr_hp, psd.psd_hp.sel(lam=lam), lw=0.9, label=rf"$\lambda = {lam}$")
ax.legend()

## Multiple cases

In [ ]:
cases = get_cases()

In [ ]:
for case in cases:
    print(case.name)
    lam = 0.2
    psd_list = []
    for sigma in [150, 198, 250, 300, 400]:
        psd = all_in_one(case.swath, lam=lam, sigma_m=sigma)
        psd["psd_hp"] = psd.psd_hp.expand_dims(sigma=[sigma])
        psd_list.append(psd)
    case.psd = xr.concat(psd_list, "sigma", data_vars=None)

In [ ]:
fig, axes = plt.subplots(
    1, 3, sharex=True, layout="constrained", figsize=(7, 2.5), dpi=200
)
fig.suptitle(rf"Varying σ ($\lambda = {lam})$")

for ax, case in zip(axes, cases):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.plot(case.psd.fr, case.psd.psd/5e4, color="k", label="Input")
    for sigma in case.psd.sigma.values:
        ax.plot(
            case.psd.fr_hp,
            case.psd.psd_hp.sel(sigma=sigma),
            lw=0.9,
            label=rf"$\sigma = {sigma}$",
        )
    ax.set_title(rf"{case.name}")
    ax.set_xlim(1, case.psd.fr_hp[-1])
    ax.set_ylim(1e-6, 1)
axes[1].legend(ncols=2)
axes[1].set_xlabel("[cycle/km]")

In [ ]:
for case in cases:
    print(case.name)
    sigma_m = 300
    psd_list = []
    for lam in [0, 0.01, 0.1, 0.2, 0.5, 1]:
        psd = all_in_one(case.swath, lam=lam, sigma_m=sigma_m)
        psd["psd_hp"] = psd.psd_hp.expand_dims(lam=[lam])
        psd_list.append(psd)
    case.psd = xr.concat(psd_list, "lam", data_vars=None)

In [ ]:
fig, axes = plt.subplots(
    1, 3, sharex=True, layout="constrained", figsize=(7, 2.5), dpi=200
)
fig.suptitle(rf"Varying λ ($\sigma = {sigma_m})$")

for ax, case in zip(axes, cases):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.plot(case.psd.fr, case.psd.psd/5e4, color="k", label="Input")
    for lam in case.psd.lam.values:
        ax.plot(
            case.psd.fr_hp,
            case.psd.psd_hp.sel(lam=lam),
            lw=0.9,
            label=rf"$\lambda = {lam}$",
        )
        ax.set_title(rf"{case.name}")
    ax.set_xlim(0.5, case.psd.fr_hp[-1])
    ax.set_ylim(None, 1)
axes[1].legend(ncols=2)

# NFFT Reconstruction

In [ ]:
case = cases_def[0]
swath = case.swath

dx, dy = get_pixel_size(swath, cross_track_margin=cross_track_margin)
span = select_span(swath, dx, dy)
span_vertices = get_span_vertices(span)

cell_ids, _, _ = healpix_geo.nested.polygon_coverage(
    span_vertices, depth=level, ellipsoid="WGS84"
)


resampled = resample(swath, cell_ids, sigma_m=275, lam=0.2)
cell_ids = resampled.cell_ids
ssh_hp = resampled.cell_data

cell_lon, cell_lat = healpix_geo.nested.healpix_to_lonlat(
    cell_ids, depth=level, ellipsoid="WGS84"
)

weights = apodize_vertices(cell_lon, cell_lat, span_vertices, length=0.07)

In [ ]:
f = ssh_hp
f = f - np.mean(f)
# f *= weights
n_frequencies = np.array([200, 200])
fx_nfft, fy_nfft, fhat = nfft(
    cell_ids,
    f,
    grid={"level": level, "indexing_scheme": "nested", "ellipsoid": "WGS84"},
    n_frequencies=n_frequencies,
)

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", dpi=200)
ax1, ax2 = axes

ims1 = hp.pcolormesh(
    cell_ids, f, healpix_grid=healpix_grid, level_tile=7, cmap="Blues_r", ax=ax1
)
ax2.pcolormesh(fx_nfft*1000, fy_nfft*1000, np.abs(fhat)**2, norm="log")
# fig.colorbar(ims1[0], ax=ax1, location="top")

set_latlon_formatters(ax1)
for ax in axes:
    ax.set_aspect("equal")

In [ ]:
from healpix_geo.auto import Grid
from pyNFFT3 import NFFT
from swot_healpix.analysis.spectra.nfft import (
    _get_coordinates,
    _get_frequencies,
    _normalize_coordinates,
)

grid = Grid(**healpix_grid)
x, y = _get_coordinates(cell_ids, grid=grid, meters=True)
xy_norm, bounds = _normalize_coordinates(x, y)
fx, fy = _get_frequencies(n_frequencies, bounds)

plan = NFFT(N=n_frequencies, M=cell_ids.size)
plan.x = np.stack(xy_norm, axis=-1)
plan.fhat = np.ascontiguousarray(fhat.T.ravel() / 235**2)
plan.nfft_trafo()

f_rec = np.real(plan.f)

In [ ]:
fig, axes = plt.subplots(1, 2, layout="constrained", sharex=True, sharey=True, dpi=200)
ax1, ax2 = axes

ims1 = hp.pcolormesh(
    cell_ids, f, healpix_grid=healpix_grid, level_tile=7, cmap="Blues_r", ax=ax1
)
ims2 = hp.pcolormesh(
    cell_ids,
    f_rec,
    healpix_grid=healpix_grid,
    level_tile=7,
    cmap="Blues_r",
    ax=ax2,
)
fig.colorbar(ims1[0], ax=ax1, location="top")
fig.colorbar(ims2[0], ax=ax2, location="top")


for ax in axes:
    ax.set_aspect("equal")
    set_latlon_formatters(ax)

In [ ]:
fig, ax = plt.subplots(layout="constrained", dpi=200)

error = f_rec - f
ims = hp.pcolormesh(
    cell_ids,
    error,
    healpix_grid=healpix_grid,
    level_tile=7,
    cmap="coolwarm",
    ax=ax,
)
fig.colorbar(ims[0], ax=ax)

# SWOT Aligned reconstruction

In [ ]:
case = cases_def[0]
swath = case.swath

dx, dy = get_pixel_size(swath, cross_track_margin=cross_track_margin)
span = select_span(swath, dx, dy)
span_vertices = get_span_vertices(span)

f = span.ssh_karin_2.values
f -= np.mean(f)

In [ ]:
fhat = np.fft.fft2(f)

fig, axes = plt.subplots(1, 2, layout="constrained", dpi=200)
ax1, ax2 = axes

im = ax1.imshow(
   f, cmap="Blues_r", origin="lower"
)
fig.colorbar(im, ax=ax1, location="top")
im = ax2.imshow(np.abs(fhat)**2, origin="lower", norm="log", vmin=1e-6)
fig.colorbar(im, ax=ax2, location="top")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
f_rec = np.real(np.fft.ifft2(fhat))

fig, axes = plt.subplots(1, 2, layout="constrained", sharex=True, sharey=True, dpi=200)
ax1, ax2 = axes

im1 = ax1.imshow(f, cmap="Blues_r", origin="lower")
im2 = ax2.imshow(f_rec, cmap="Blues_r", origin="lower")
fig.colorbar(im1, ax=ax1, location="top")
fig.colorbar(im2, ax=ax2, location="top")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
fig, ax = plt.subplots(layout="constrained", dpi=200)

error = f_rec - f
im = ax.imshow(error, cmap="coolwarm", origin="lower")
fig.colorbar(im, ax=ax)

## Using NFFT (on regular grid)

In [ ]:
n_frequencies = np.array([160, 160])
plan = NFFT(N=n_frequencies, M=f.size, n=np.array([1024, 1024]), m=20)
x = np.linspace(-0.5, 0.5, 160)
y = np.linspace(-0.5, 0.5, 164)
xy = np.stack(np.meshgrid(x, y), axis=-1).reshape(-1, 2)
plan.x = xy
plan.f = np.ascontiguousarray(f.ravel().astype("complex128"))
plan.adjoint()
fhat = plan.fhat.copy()
fhat = fhat.reshape(n_frequencies).T

fig, axes = plt.subplots(1, 2, layout="constrained", dpi=200)
ax1, ax2 = axes

im = ax1.imshow(
   f, cmap="Blues_r", origin="lower"
)
ax2.imshow(np.abs(fhat)**2, origin="lower", norm="log")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
plan = NFFT(N=n_frequencies, M=f.size, n=np.array([1024, 1024]), m=20)
plan.x = xy
plan.fhat = fhat.T.ravel()
plan.nfft_trafo()
f_rec = np.real(plan.f.copy())
f_rec = f_rec.reshape(f.shape) / (160*160)

fig, axes = plt.subplots(1, 2, layout="constrained", sharex=True, sharey=True, dpi=200)
ax1, ax2 = axes

im1 = ax1.imshow(f, cmap="Blues_r", origin="lower")
im2 = ax2.imshow(f_rec, cmap="Blues_r", origin="lower", vmax=11)
fig.colorbar(im1, ax=ax1, location="top")
fig.colorbar(im2, ax=ax2, location="top")

for ax in axes:
    ax.set_aspect("equal")

In [ ]:
fig, ax = plt.subplots(layout="constrained", dpi=200)

error = f_rec - f
im = ax.imshow(error[1:-1, 1:-1], cmap="coolwarm", origin="lower")
fig.colorbar(im, ax=ax)